---
# Import Library

---

In [1]:
# Processing and Displying Data in DataFrame form.
import pandas as pd

# Calculate Data.
import numpy as np

# Spliting Data into Train and Test
from sklearn.model_selection import train_test_split

# handling Outlier.
from feature_engine.outliers import Winsorizer

# Scaling Data. (Numeric - Skew)
# from sklearn.preprocessing import MinMaxScaler

# Encode Data. (Categorical - Nominal)
# from sklearn.preprocessing import OneHotEncoder

# Colleration Check each column.
# import phik

# Machine Learning Model.
# from sklearn.linear_model import LogisticRegression
# from sklearn.tree import DecisionTreeClassifier #TODO kalau udah RF, DT gaperlu lagi
# from sklearn.ensemble import RandomForestClassifier

# Calculate, compare, visualize Predict Score.
# from sklearn.metrics import recall_score, confusion_matrix, ConfusionMatrixDisplay

# Model Saving.
# import pickle

# Ignoring Warning from Python.
import warnings
warnings.filterwarnings("ignore")

---
# Data Loading

---

In [2]:
# Using file from cleaned Folder outside .ipynb path file
Data = pd.read_csv('../data/cleaned/order_agg_modeling_v0_1.csv', index_col=0)
Backup_data = Data.copy()

In [3]:
# to Shown all column.
pd.options.display.max_columns = None

# check summary of data
Data.info()

# top 5 Data Sample.
Data.head()

<class 'pandas.core.frame.DataFrame'>
Index: 98832 entries, 0 to 99440
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       98832 non-null  object 
 1   customer_id                    98832 non-null  object 
 2   order_status                   98832 non-null  object 
 3   order_purchase_timestamp       98832 non-null  object 
 4   order_approved_at              98672 non-null  object 
 5   order_delivered_carrier_date   97658 non-null  object 
 6   order_delivered_customer_date  96476 non-null  object 
 7   order_estimated_delivery_date  98832 non-null  object 
 8   total_items                    98660 non-null  float64
 9   total_order_value              98660 non-null  float64
 10  total_freight                  98660 non-null  float64
 11  avg_item_price                 98660 non-null  float64
 12  payment_type                   98831 non-null  obje

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_items,total_order_value,total_freight,avg_item_price,payment_type,total_payment_value,max_installments,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_canceled
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,29.99,8.72,29.99,voucher,38.71,1.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,118.70,22.76,118.70,boleto,141.46,1.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,159.90,19.22,159.90,credit_card,179.12,3.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,45.00,27.20,45.00,credit_card,72.20,1.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,19.90,8.72,19.90,credit_card,28.62,1.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,0


By above output, all of the column data type in the dataset behave as they shold be. However, note that the dataset still contains some missing values. We will handle these missing values later before modeling process further.

---
# Feature Engineering

---

---
## Duplicated Check

---

In [4]:
print(f"Duplicated Data  : {Data.duplicated().sum()}")
print(f"Total Data : {Data.shape[0]}")

Duplicated Data  : 0
Total Data : 98832


Based on the duplicate check, no duplicated records were found in the dataset.

---

---
## Missing value Check

---

In [5]:
Data.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1174
order_delivered_customer_date    2356
order_estimated_delivery_date       0
total_items                       172
total_order_value                 172
total_freight                     172
avg_item_price                    172
payment_type                        1
total_payment_value                 1
max_installments                    1
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
is_canceled                         0
dtype: int64

Based on the Missing Value check, there are some columns that need to be handled.

---

---
## Missing Value Handling

---
In this section, only handle some missing values that not including impute process with mean value for numerical and modus for categorical variables. Moreover, missing value handling that includes impute process with mean value or modus value (that needs to be based on train set only, to prevent data leakage) will be done later after train-test split.

In [6]:
Data_Processed = Data.drop(columns=['order_approved_at'
                        ,'order_delivered_carrier_date'
                        ,'order_delivered_customer_date'
                        ,'order_estimated_delivery_date']
                ,axis=1 )

Note that we concerned for the cancellation rate and want to predict whether the order will be cancelled or not. In the future, we will notify (by system) the warning of such order will be cancelled prior the order approved by the system. Hence, the objective of the model is to predict the cancellation rate **before** the transaction is approved, which impact to the approval date column and all subsequent activity (after transaction is approved) date columns will be dropped.

---

In [7]:
Missing_Data = Data_Processed[Data_Processed['payment_type'].isnull()].index
Data_Processed.loc[Missing_Data]

,order_id,customer_id,order_status,order_purchase_timestamp,total_items,total_order_value,total_freight,avg_item_price,payment_type,total_payment_value,max_installments,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_canceled
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,3.0,134.97,8.49,44.99,NaN,NaN,NaN,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,0


In [8]:
Data_Processed = Data_Processed.dropna(subset=['payment_type','max_installments','total_payment_value'],axis=0)

TODO   
[Need Finalize] Reasoning :   
The missing values in the `payment_type`, `total_payment_value`, and `max_installments` columns occur in the same row and appear independent of other columns. Because only one record is affected, the row will be removed.

---

In [9]:
Data_Processed['total_items'] = Data_Processed['total_items'].fillna(1)
Data_Processed['total_order_value'] = Data_Processed['total_order_value'].fillna(Data_Processed['total_payment_value'])
Data_Processed['total_freight'] = Data_Processed['total_freight'].fillna(0)
Data_Processed['avg_item_price'] = Data_Processed['avg_item_price'].fillna(Data_Processed['total_payment_value'])

TODO   
[need finalize]Reasoning :   
Missing values in `total_items` are imputed with `1` under the assumption that **each order must contain at least one item**. Since **an order cannot exist without any item**.

For `total_order_value` and `avg_item_price` are imputed using `total_payment_value`, with `total_items` are inputed as 1, the `total_order_value` and `avg_item_price` are equivalent to the total amount paid.

For `total_freight` are imputed with `0` as default because min value for `total_freight` is `0` that indicate as **free shipping**.

---

In [10]:
Data_Processed.isnull().sum()

order_id                    0
customer_id                 0
order_status                0
order_purchase_timestamp    0
total_items                 0
total_order_value           0
total_freight               0
avg_item_price              0
payment_type                0
total_payment_value         0
max_installments            0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
is_canceled                 0
dtype: int64

Note that after all above process, no missing value records were found in the dataset. Therefore, there will be no process of missing value handling that requires impute with mean (for numerical) and modus (for categorical) value after train-test split. We will move forward to the next step, split the column X (features) & y (target) from the dataset.

---

---
## Split column X & y

---

In [11]:
# Excluding column that like primary key, candidate primary key and looks like Target column
X = Data_Processed.drop(columns=['customer_id','order_id','customer_unique_id','order_status','is_canceled'],axis=1)

# using is_canceled column as predict target 
y = Data_Processed.is_canceled

In [12]:
X.head()

,order_purchase_timestamp,total_items,total_order_value,total_freight,avg_item_price,payment_type,total_payment_value,max_installments,customer_zip_code_prefix,customer_city,customer_state
0,2017-10-02 10:56:33,1.0,29.99,8.72,29.99,voucher,38.71,1.0,3149,sao paulo,SP
1,2018-07-24 20:41:37,1.0,118.70,22.76,118.70,boleto,141.46,1.0,47813,barreiras,BA
2,2018-08-08 08:38:49,1.0,159.90,19.22,159.90,credit_card,179.12,3.0,75265,vianopolis,GO
3,2017-11-18 19:28:06,1.0,45.00,27.20,45.00,credit_card,72.20,1.0,59296,sao goncalo do amarante,RN
4,2018-02-13 21:18:39,1.0,19.90,8.72,19.90,credit_card,28.62,1.0,9195,santo andre,SP


In [13]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98831 entries, 0 to 99440
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_purchase_timestamp  98831 non-null  object 
 1   total_items               98831 non-null  float64
 2   total_order_value         98831 non-null  float64
 3   total_freight             98831 non-null  float64
 4   avg_item_price            98831 non-null  float64
 5   payment_type              98831 non-null  object 
 6   total_payment_value       98831 non-null  float64
 7   max_installments          98831 non-null  float64
 8   customer_zip_code_prefix  98831 non-null  int64  
 9   customer_city             98831 non-null  object 
 10  customer_state            98831 non-null  object 
dtypes: float64(6), int64(1), object(4)
memory usage: 9.0+ MB


By above process we have split the dataset into `X` (features) and `y` (target).

---
## Cardinality Check

---

In [ ]:
category = ['payment_type','max_installments','customer_zip_code_prefix','customer_city','customer_state']

for i in X[category]:
    print(f'Number of unique value of column `{i}` : {X[i].nunique()}')
    print(f'Unique values of column `{i}` : {X[i].unique()}')
    print('')

Jumlah unique value dari kolom payment_type : 5
Unique value dari kolom payment_type : ['voucher' 'boleto' 'credit_card' 'debit_card' 'not_defined']

Jumlah unique value dari kolom max_installments : 24
Unique value dari kolom max_installments : [ 1.  3.  6. 10.  4.  2.  8.  9.  7.  5. 13. 12. 15. 14. 21. 18. 24. 17.
 11. 20. 23. 16. 22.  0.]

Jumlah unique value dari kolom customer_zip_code_prefix : 14977
Unique value dari kolom customer_zip_code_prefix : [ 3149 47813 75265 ... 83870  5127 45920]

Jumlah unique value dari kolom customer_city : 4111
Unique value dari kolom customer_city : ['sao paulo' 'barreiras' 'vianopolis' ... 'messias targino'
 'campo do tenente' 'nova vicosa']

Jumlah unique value dari kolom customer_state : 27
Unique value dari kolom customer_state : ['SP' 'BA' 'GO' 'RN' 'PR' 'RS' 'RJ' 'MG' 'SC' 'RR' 'PE' 'TO' 'CE' 'DF'
 'SE' 'MT' 'PB' 'PA' 'RO' 'ES' 'AP' 'MS' 'MA' 'PI' 'AL' 'AC' 'AM']



Based on the Cardinality Check result above, `max_installments`, `customer_zip_code_prefix`, `customer_city`, and `customer_state` are categorized as high-cardinality features and will require appropriate handling.

---

---
## Cardinality handling

---
#TODO menurut ku cuma 1 section aja, yang di atas itu Cardinality Handling, yang di sini gausah. Tapi bebas, bahas di discord saja

#### Handling column `max_installments`:

In [ ]:
# create binning of `max_installments` column
X['installment_bin'] = pd.cut(
    X['max_installments']
    ,bins=[0, 6, 12, 18, 24] # 5 binning
    ,labels=['very_short','short','medium','long']
    ,include_lowest=True
)
X = X.drop('max_installments',axis=1)

In [16]:
X['installment_bin'].unique()

['very_short', 'short', 'medium', 'long']
Categories (4, object): ['very_short' < 'short' < 'medium' < 'long']

TODO   
[need finalize] Reasoning :   
The `max_installments` column cardinality is reduced through a binning process to minimize overfitting risk and enhance model stability.

---

#### Handling column `customer_state`:

In [17]:
Top_9_state = X['customer_state'].value_counts().nlargest(9).index

X['customer_state'] = np.where(X['customer_state'].isin(Top_9_state)
                               ,X['customer_state'] # True Result
                               ,'Other') #False Result

X['customer_state'].value_counts()


customer_state
SP       41453
RJ       12784
MG       11560
Other    11453
RS        5442
PR        5005
SC        3619
BA        3360
DF        2128
ES        2027
Name: count, dtype: int64

TODO   
[need finalize] Reasoning :   
The `customer_state` column cardinality is reduced by retaining the **top 9 states** and grouping the remaining states into an `“Other”` category to minimize overfitting risk and enhance model stability.

---

In [18]:
X = X.drop(columns=['customer_zip_code_prefix','customer_city'],axis=1)

TODO   
Reasoning :   
The `customer_zip_code_prefix` and `customer_city` information is overlapping with `customer_state`. These columns will be removed to avoid redundancy and high cardinality issues.

---

---
## Feature Creation

---

#TODO perlu tulis reasoning singkat nya

In [19]:
X['order_purchase_timestamp'] = pd.to_datetime(X['order_purchase_timestamp'])
X['days_of_week'] = X['order_purchase_timestamp'].dt.dayofweek
X['is_weekend'] = X['days_of_week'] >= 5
X['is_weekend'] = X['is_weekend'].astype(int)
X = X.drop(columns=['days_of_week','order_purchase_timestamp'], axis=1)
X.head()

,total_items,total_order_value,total_freight,avg_item_price,payment_type,total_payment_value,customer_state,installment_bin,is_weekend
0,1.0,29.99,8.72,29.99,voucher,38.71,SP,very_short,0
1,1.0,118.70,22.76,118.70,boleto,141.46,BA,very_short,0
2,1.0,159.90,19.22,159.90,credit_card,179.12,Other,very_short,0
3,1.0,45.00,27.20,45.00,credit_card,72.20,Other,very_short,1
4,1.0,19.90,8.72,19.90,credit_card,28.62,SP,very_short,0


---
## Spliting categorical and numerical columns

---

In [20]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98831 entries, 0 to 99440
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   total_items          98831 non-null  float64 
 1   total_order_value    98831 non-null  float64 
 2   total_freight        98831 non-null  float64 
 3   avg_item_price       98831 non-null  float64 
 4   payment_type         98831 non-null  object  
 5   total_payment_value  98831 non-null  float64 
 6   customer_state       98831 non-null  object  
 7   installment_bin      98831 non-null  category
 8   is_weekend           98831 non-null  int64   
dtypes: category(1), float64(5), int64(1), object(2)
memory usage: 6.9+ MB


In [21]:
cat_col = ['payment_type','customer_state','installment_bin','is_weekend']
num_col = ['total_items','total_order_value','total_freight','avg_item_price','total_payment_value']

---
## Train-Test Split

---

In [22]:
# memisahkan data menjadi Train & Test dengan persentase 70% Train dan 30% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify=y, random_state = 50)

# menampilkan banyaknya data Train
print(f"Train Data Size : {X_train.shape[0]}\n")

# menampilkan banyaknya data Test
print(f"Test Data Size : {X_test.shape[0]}")

Train Data Size : 69181

Test Data Size : 29650


---
## Outliers Handling

---

In [23]:
X_train[num_col].describe()

,total_items,total_order_value,total_freight,avg_item_price,total_payment_value
count,69181.000000,69181.000000,69181.000000,69181.000000,69181.000000
mean,1.141816,137.959883,22.777400,126.088133,160.763154
std,0.540117,214.096666,21.950849,191.780403,223.817434
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,45.950000,13.800000,42.000000,62.010000
50%,1.000000,86.790000,17.150000,79.000000,105.280000
75%,1.000000,149.900000,23.990000,139.900000,176.900000
max,20.000000,13440.000000,1794.960000,6735.000000,13664.080000


TODO   
Reasoning :   
Exclude kolom `total_items` karena nilai Q1 & Q3 sama

---

In [24]:
def check_skewness(df, *column_names):
    return {col: df[col].skew() for col in column_names if col in df.columns}

In [25]:
skewness_results = check_skewness(X_train, 'total_order_value','total_freight','avg_item_price','total_payment_value')

for col, skewness in skewness_results.items():
    print(f"{col}: {skewness}")

total_order_value: 10.786602639901416
total_freight: 13.978816282019974
avg_item_price: 7.945994928124064
total_payment_value: 10.221626681503642


In [26]:
extreme_skewed_columns = ['total_order_value','total_freight','avg_item_price','total_payment_value']
print(f"Extreme Skewed: {extreme_skewed_columns}")

Extreme Skewed: ['total_order_value', 'total_freight', 'avg_item_price', 'total_payment_value']


In [27]:
# Pengecekan Persentase Outlier setiap kolom 
def calculate_outlier_percentages_skew(df, columns, distance):
    for variable in columns:
        IQR = df[variable].quantile(0.75) - df[variable].quantile(0.25)
        lower_boundary = df[variable].quantile(0.25) - (IQR * distance)
        upper_boundary = df[variable].quantile(0.75) + (IQR * distance)

        outliers = df[(df[variable] < lower_boundary) | (df[variable] > upper_boundary)]
        outlier_percentage = len(outliers) / len(df) * 100

        print('Percentage of outliers in {}: {:.2f}%'.format(variable, outlier_percentage))

# Menampilkan persentase outlier pada kolom yang Skewnessnya extreme
print(calculate_outlier_percentages_skew(X_train, extreme_skewed_columns, 3))

Percentage of outliers in total_order_value: 4.18%
Percentage of outliers in total_freight: 5.08%
Percentage of outliers in avg_item_price: 3.83%
Percentage of outliers in total_payment_value: 4.02%
None


In [28]:
# Function untuk handling outlier dengan metode capping pada kolom dengan skewness extreme
def apply_winsorization_skew(train, variables, capping_method='iqr', tail='both', fold=3):
    winsoriser = Winsorizer(capping_method=capping_method, tail=tail, fold=fold, variables=variables)
    train_capped = winsoriser.fit_transform(train)
    return train_capped

# Apply to X_train column
X_train = apply_winsorization_skew(X_train, extreme_skewed_columns)

In [29]:
print(calculate_outlier_percentages_skew(X_train, extreme_skewed_columns, 3))

Percentage of outliers in total_order_value: 0.00%
Percentage of outliers in total_freight: 0.00%
Percentage of outliers in avg_item_price: 0.00%
Percentage of outliers in total_payment_value: 0.00%
None
